<a href="https://colab.research.google.com/github/attabeezy/crop-guard/blob/main/notebooks/01_data_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CropGuard — CCMT data setup and inspection

This notebook downloads the public CCMT Ghana crop pest and disease dataset, locates image folders, reports counts, and displays random examples. It does **not** train a model or upload dataset files to GitHub.

Authoritative source: [Mendeley Data, DOI 10.17632/bwh3zbpkpv.1](https://data.mendeley.com/datasets/bwh3zbpkpv/1), licensed CC BY 4.0. We will use the original/raw images, not the pre-augmented set.

## 1. Install the downloader

Kaggle hosts a convenient mirror of the Mendeley dataset. `kagglehub` downloads public datasets into Colab's temporary storage.

In [ ]:
%pip install -q kagglehub

## 2. Download CCMT

If Kaggle asks for authentication, use Colab's **Secrets** panel to add `KAGGLE_USERNAME` and `KAGGLE_KEY`, then rerun this cell.

In [ ]:
from pathlib import Path
import kagglehub

DATASET_HANDLE = "nirmalsankalana/crop-pest-and-disease-detection"
dataset_root = Path(kagglehub.dataset_download(DATASET_HANDLE))
print(f"Dataset downloaded to: {dataset_root}")

## 3. Inspect the archive structure

In [ ]:
print("Top-level entries:")
for path in sorted(dataset_root.iterdir()):
    print("-", path.name)

In [ ]:
from collections import Counter

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
images = [
    path for path in dataset_root.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
]

folder_counts = Counter(str(path.parent.relative_to(dataset_root)) for path in images)
print(f"Total image files found: {len(images):,}\n")
for folder, count in sorted(folder_counts.items()):
    print(f"{count:7,}  {folder}")

The official raw-image totals are approximately 6,549 cashew, 7,508 cassava, 5,389 maize, and 5,435 tomato images. A total near 102,976 indicates the augmented data and must not be used for our honest split.

## 4. View random examples

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

random.seed(2026)
samples = random.sample(images, min(12, len(images)))
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for axis, path in zip(axes.flat, samples):
    with Image.open(path) as source:
        axis.imshow(ImageOps.exif_transpose(source).convert("RGB"))
    axis.set_title(str(path.relative_to(dataset_root)), fontsize=8)
    axis.axis("off")

for axis in axes.flat[len(samples):]:
    axis.axis("off")

plt.tight_layout()

## 5. Check the expected classes

CropGuard's locked scope is cashew, cassava, and maize. The expected 17 classes are listed below. Folder spelling may differ, so inspect the count table before reorganizing anything.

In [ ]:
expected_classes = {
    "cashew": ["anthracnose", "gummosis", "healthy", "leaf miner", "red rust"],
    "cassava": ["bacterial blight", "brown spot", "green mite", "healthy", "mosaic"],
    "maize": ["fall armyworm", "grasshopper", "healthy", "leaf beetle", "leaf blight", "leaf spot", "streak virus"],
}

for crop, classes in expected_classes.items():
    print(f"{crop.title()} ({len(classes)} classes): {', '.join(classes)}")

## Next step

Run all cells, then save or share the folder-count output. The preparation script will be adjusted to the archive's verified raw-folder layout before creating train, validation, and test manifests.